
# V-JEPA2 SAILS Fine-tuning Across CV Folds (Top-k + Subset Analysis)

This notebook runs the finetuning pipeline across all CV folds (fresh checkpoint per fold), reports top-1/top-2 metrics, precision/recall/F1 (micro/macro/weighted + per-class), plots PR curves, and breaks down performance on specific clip subsets (single-child vs multi-child videos, videos with mixed RMM labels, and quality buckets from the ratings sheet). All metrics are computed at the clip level.


In [1]:

from pathlib import Path
from functools import partial
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu
from tqdm import tqdm
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
)
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor

try:
    import wandb  # Optional
except ImportError:
    wandb = None

# Paths
csv_dir = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/cv_folds')
clips_root = Path('/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips')
output_root = Path('runs/vjepa2_rmm_type_cv')

# Folds (train/val CSVs)
train_csvs = sorted(csv_dir.glob('fold_*_train.csv'))
val_csvs = sorted(csv_dir.glob('fold_*_val.csv'))
assert len(train_csvs) == len(val_csvs) and len(train_csvs) > 0, 'Missing fold CSVs'

# Hyperparameters
model_id = 'facebook/vjepa2-vitl-fpc16-256-ssv2'
batch_size = 1
num_workers = 8
num_epochs = 10
lr = 1e-5
accumulation_steps = 4
log_interval = 50
wandb_mode = 'online'  # online|offline|disabled

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if wandb_mode != 'disabled' and wandb is None:
    print('⚠️ wandb not installed; set wandb_mode="disabled" or pip install wandb to log runs.')


Using device: cuda


In [2]:

# Helpers: loading splits, quality buckets, and metadata

def quality_bucket(val):
    try:
        x = float(val)
    except (TypeError, ValueError):
        return 'unknown'
    if x >= 4:
        return 'high'
    if x >= 3:
        return 'medium'
    return 'low'


def load_split(csv_paths):
    records = []
    missing = []
    for csv_path in csv_paths:
        stem = csv_path.stem
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            seg = row.get('segment_id') or row.get('segment_global_id')
            label = row.get('rmm_type')
            if pd.isna(seg) or pd.isna(label):
                continue
            clip_path = clips_root / stem / f"{seg}.mp4"
            if not clip_path.exists():
                missing.append((seg, clip_path))
                continue
            rec = {
                'segment_id': seg,
                'label': str(label),
                'clip': clip_path,
                'video_id': row.get('video_id'),
                'n_children': pd.to_numeric(row.get('n_children'), errors='coerce'),
                'n_adults': pd.to_numeric(row.get('n_adults'), errors='coerce'),
                'quality_rating': row.get('quality_rating'),
            }
            records.append(rec)
    return records, missing


# Build global label dictionary and video->label counts for mixed-label detection
all_records, _ = load_split(train_csvs + val_csvs)
video_label_counts = {}
for rec in all_records:
    video_id = rec.get('video_id')
    if video_id is None:
        continue
    video_label_counts.setdefault(video_id, set()).add(rec['label'])
video_label_counts = {k: len(v) for k, v in video_label_counts.items()}

all_labels = sorted({r['label'] for r in all_records})
label2id = {lbl: i for i, lbl in enumerate(all_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}
print('Labels:', label2id)


Labels: {'hands flapping': 0, 'jumping': 1, 'one hand flap': 2, 'rocking': 3, 'spinning': 4}


In [3]:

# Dataset + collate

class RMMDataset(Dataset):
    def __init__(self, records, label2id, frames_per_clip, video_label_counts):
        self.records = records
        self.label2id = label2id
        self.frames_per_clip = frames_per_clip
        self.video_label_counts = video_label_counts

    def __len__(self):
        return len(self.records)

    def _sample_indices(self, vr):
        total = len(vr)
        if total <= 0:
            return np.zeros(self.frames_per_clip, dtype=np.int64)
        return np.round(np.linspace(0, total - 1, self.frames_per_clip)).astype('int64')

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            vr = VideoReader(str(rec['clip']), ctx=cpu(0))
            indices = self._sample_indices(vr)
            frames = vr.get_batch(indices).asnumpy()
        except Exception as exc:
            print(f"[bad clip] {rec['clip']}: {exc}")
            return None

        label_id = self.label2id[rec['label']]
        video_id = rec.get('video_id')
        meta = {
            'segment_id': rec.get('segment_id'),
            'video_id': video_id,
            'label_name': rec['label'],
            'n_children': rec.get('n_children'),
            'n_adults': rec.get('n_adults'),
            'quality_rating': rec.get('quality_rating'),
            'quality_bucket': quality_bucket(rec.get('quality_rating')),
            'mixed_video': video_id in self.video_label_counts and self.video_label_counts[video_id] > 1,
        }
        return frames, label_id, meta


def collate_fn(samples, processor):
    samples = [s for s in samples if s is not None]
    if not samples:
        return None, None, None
    frame_batches, labels, metas = zip(*samples)
    inputs = processor(list(frame_batches), return_tensors='pt')
    labels_tensor = torch.tensor(labels)
    return inputs, labels_tensor, list(metas)


In [4]:

# Eval helpers

def aggregate_preds(labels, preds, probs, metas):
    df = pd.DataFrame(metas)
    df['label_id'] = labels
    df['pred_top1'] = preds
    df['label_name'] = df['label_id'].map(id2label)
    df['pred_name'] = df['pred_top1'].map(id2label)
    return df


def evaluate(loader, model, device, collect_probs=False):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels, all_metas = [], [], []
    all_probs = []
    with torch.no_grad():
        for inputs, labels, metas in loader:
            if inputs is None or labels is None:
                continue
            labels = labels.to(device)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = logits.argmax(-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_metas.extend(metas)
            if collect_probs:
                all_probs.append(torch.softmax(logits, dim=-1).cpu())
    acc = correct / max(total, 1)
    probs_tensor = torch.cat(all_probs, dim=0) if all_probs else None
    return acc, all_preds, all_labels, all_metas, probs_tensor


def evaluate_topk(loader, model, device, k=2):
    model.eval()
    correct_top1 = 0
    correct_topk = 0
    total = 0

    all_preds_top1 = []
    all_preds_topk = []
    all_labels = []
    all_probs = []
    all_metas = []

    with torch.no_grad():
        for inputs, labels, metas in loader:
            if inputs is None or labels is None:
                continue

            labels = labels.to(device)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)

            preds_top1 = logits.argmax(-1)
            correct_top1 += (preds_top1 == labels).sum().item()

            _, topk_indices = torch.topk(probs, k, dim=-1)
            correct_topk += torch.any(topk_indices == labels.unsqueeze(-1), dim=-1).sum().item()

            total += labels.size(0)

            all_preds_top1.extend(preds_top1.cpu().tolist())
            all_preds_topk.append(topk_indices.cpu())
            all_labels.extend(labels.cpu().tolist())
            all_probs.append(probs.cpu())
            all_metas.extend(metas)

    top1_acc = correct_top1 / max(total, 1)
    topk_acc = correct_topk / max(total, 1)

    all_preds_topk = torch.cat(all_preds_topk, dim=0) if all_preds_topk else torch.tensor([])
    all_probs = torch.cat(all_probs, dim=0) if all_probs else torch.tensor([])

    return {
        'top1_acc': top1_acc,
        f'top{k}_acc': topk_acc,
        'improvement': topk_acc - top1_acc,
        'preds_top1': all_preds_top1,
        'preds_topk': all_preds_topk,
        'labels': all_labels,
        'probs': all_probs,
        'metas': all_metas,
    }


def compute_classification_metrics(labels, preds, probs, id2label):
    class_names = [id2label[i] for i in range(len(id2label))]
    report = classification_report(labels, preds, target_names=class_names, output_dict=True, zero_division=0)
    prec_rec_f1 = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    per_class = pd.DataFrame({
        'class': class_names,
        'precision': prec_rec_f1[0],
        'recall': prec_rec_f1[1],
        'f1': prec_rec_f1[2],
        'support': prec_rec_f1[3],
    })

    # PR curves for top classes by support
    y_true = np.array(labels)
    pr_curves = {}
    if probs is not None and len(class_names) > 0:
        supports = per_class.set_index('class')['support']
        top_classes = supports.sort_values(ascending=False).head(min(5, len(class_names))).index.tolist()
        for cls in top_classes:
            cls_id = class_names.index(cls)
            y_bin = (y_true == cls_id).astype(int)
            scores = probs[:, cls_id]
            prec, rec, _ = precision_recall_curve(y_bin, scores)
            ap = average_precision_score(y_bin, scores)
            pr_curves[cls] = {'precision': prec, 'recall': rec, 'ap': ap}
    summary = {
        'macro_f1': report['macro avg']['f1-score'],
        'macro_precision': report['macro avg']['precision'],
        'macro_recall': report['macro avg']['recall'],
        'micro_f1': report['micro avg']['f1-score'],
        'micro_precision': report['micro avg']['precision'],
        'micro_recall': report['micro avg']['recall'],
        'weighted_f1': report['weighted avg']['f1-score'],
    }
    return summary, per_class, pr_curves


def metrics_for_subset(mask, labels, preds, probs, id2label):
    if mask.sum() == 0:
        return None
    l = labels[mask]
    p = preds[mask]
    pr = probs[mask] if probs is not None else None
    summary, per_class, _ = compute_classification_metrics(l, p, pr, id2label)
    summary['size'] = len(l)
    return summary


In [5]:

# Training per fold

processor = VJEPA2VideoProcessor.from_pretrained(model_id)
frames_per_clip = (
    getattr(processor, 'num_frames', None)
    or getattr(getattr(processor, 'image_processor', processor), 'num_frames', None)
    or getattr(getattr(processor, 'feature_extractor', processor), 'num_frames', None)
    or getattr(getattr(processor, 'config', {}), 'num_frames', None)
    or getattr(getattr(processor, 'config', {}), 'frames_per_clip', None)
    or 32
)
frames_per_clip = 32  # explicit override used previously
print('Frames per clip:', frames_per_clip)

fold_results = []

for fold_idx, (train_csv, val_csv) in enumerate(zip(train_csvs, val_csvs)):
    print(f"\n {'='*80} Fold {fold_idx}: train={train_csv.name}, val={val_csv.name} {'='*80}")
    train_records, miss_train = load_split([train_csv])
    val_records, miss_val = load_split([val_csv])
    print(' Missing train clips:', len(miss_train), '| Missing val clips:', len(miss_val))

    train_ds = RMMDataset(train_records, label2id, frames_per_clip, video_label_counts)
    val_ds = RMMDataset(val_records, label2id, frames_per_clip, video_label_counts)

    collate = partial(collate_fn, processor=processor)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
        prefetch_factor=2 if num_workers > 0 else None,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
        prefetch_factor=2 if num_workers > 0 else None,
    )

    model = VJEPA2ForVideoClassification.from_pretrained(
        model_id,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True,
    ).to(device)

    for param in model.vjepa2.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    run_name = f'fold{fold_idx}-vjepa2-{frames_per_clip}fr'
    wandb_enabled = wandb_mode != 'disabled' and wandb is not None
    if wandb_enabled:
        wandb.init(project='vjepa-rmm', name=run_name, config={'lr': lr, 'batch_size': batch_size}, mode=wandb_mode)

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()
        running_loss = 0.0
        num_batches = 0

        for step, (inputs, labels, metas) in enumerate(train_loader, start=1):
            if inputs is None or labels is None:
                continue
            labels = labels.to(device)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss / accumulation_steps
            loss.backward()
            running_loss += loss.item()
            num_batches += 1

            if step % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

            if step % log_interval == 0:
                avg_loss = running_loss / max(num_batches, 1) * accumulation_steps
                print(f'  Epoch {epoch} Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}')

            if wandb_enabled:
                wandb.log({'train/loss': loss.item(), 'epoch': epoch, 'step': step})

        if num_batches and num_batches % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()

        val_acc, _, _, _, _ = evaluate(val_loader, model, device)
        if wandb_enabled:
            wandb.log({'val/acc': val_acc, 'epoch': epoch})
        print(f'  Epoch {epoch} complete | val_acc={val_acc:.3f}')

    # Save checkpoint per fold
    fold_out = output_root / f'fold_{fold_idx}'
    fold_out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(fold_out)
    processor.save_pretrained(fold_out)

    # Full eval with top-k and metrics
    topk = evaluate_topk(val_loader, model, device, k=2)
    labels_arr = np.array(topk['labels'])
    preds_arr = np.array(topk['preds_top1'])
    probs_arr = topk['probs'].numpy() if isinstance(topk['probs'], torch.Tensor) else None
    metas_df = aggregate_preds(labels_arr, preds_arr, probs_arr, topk['metas'])

    summary_metrics, per_class_df, pr_curves = compute_classification_metrics(labels_arr, preds_arr, probs_arr, id2label)

    # Subset masks (clip-level)
    masks = {
        'n_children_1': metas_df['n_children'] == 1,
        'n_children_gt1': metas_df['n_children'] > 1,
        'mixed_video': metas_df['mixed_video'] == True,
    }
    for qb in metas_df['quality_bucket'].dropna().unique():
        masks[f'quality_{qb}'] = metas_df['quality_bucket'] == qb

    subset_metrics = {}
    for name, mask in masks.items():
        res = metrics_for_subset(mask.values, labels_arr, preds_arr, probs_arr, id2label)
        if res:
            subset_metrics[name] = res

    # Store fold summary
    fold_results.append({
        'fold': fold_idx,
        'train_csv': train_csv.name,
        'val_csv': val_csv.name,
        'top1_acc': topk['top1_acc'],
        'top2_acc': topk['top2_acc'],
        'improvement': topk['improvement'],
        **summary_metrics,
        'per_class': per_class_df,
        'pr_curves': pr_curves,
        'subset_metrics': subset_metrics,
        'metas': metas_df,
    })

    if wandb_enabled:
        # Log scalar metrics
        wandb.log({
            'eval/top1_acc': topk['top1_acc'],
            'eval/top2_acc': topk['top2_acc'],
            'eval/top2_improvement': topk['improvement'],
            'eval/macro_f1': summary_metrics['macro_f1'],
            'eval/micro_f1': summary_metrics['micro_f1'],
            'eval/weighted_f1': summary_metrics['weighted_f1'],
        })
        # Log per-class table
        wandb.log({'eval/per_class': wandb.Table(dataframe=per_class_df)})
        # Log subset metrics table
        if subset_metrics:
            subset_rows = []
            for name, metrics in subset_metrics.items():
                row = {'subset': name}
                row.update(metrics)
                subset_rows.append(row)
            wandb.log({'eval/subsets': wandb.Table(dataframe=pd.DataFrame(subset_rows))})
        wandb.finish()

    print(f"Fold {fold_idx} | top1={topk['top1_acc']:.3f} | top2={topk['top2_acc']:.3f} | macro F1={summary_metrics['macro_f1']:.3f}")


Frames per clip: 32

 ================================================================================ Fold 0: train=fold_0_train.csv, val=fold_0_val.csv ================================================================================
 Missing train clips: 0 | Missing val clips: 0


/home/brukew/miniconda3/envs/vjepa2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Some weights of VJEPA2ForVideoClassification were not initialized from the model checkpoint at facebook/vjepa2-vitl-fpc16-256-ssv2 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([174]) in the checkpoint and torch.Size([5]) in the model instantiated
- classifier.weight: found shape torch.Size([174, 1024]) in the checkpoint and torch.Size([5, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predi

/home/brukew/miniconda3/envs/vjepa2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


  Epoch 1 Step 50/349 | Loss: 1.5939
  Epoch 1 Step 100/349 | Loss: 1.6008
  Epoch 1 Step 150/349 | Loss: 1.5914
  Epoch 1 Step 200/349 | Loss: 1.6111
  Epoch 1 Step 250/349 | Loss: 1.6054
  Epoch 1 Step 300/349 | Loss: 1.5477
[bad clip] /orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips/fold_0_val/L4K0P3T4Q8_36_month_185_0_0.mp4: [13:53:27] /github/workspace/src/video/video_reader.cc:151: Check failed: st_nb >= 0 (-1381258232 vs. 0) ERROR cannot find video stream with wanted index: -1
  Epoch 1 complete | val_acc=0.448
  Epoch 2 Step 50/349 | Loss: 1.3119
  Epoch 2 Step 100/349 | Loss: 1.3729
  Epoch 2 Step 150/349 | Loss: 1.4068
  Epoch 2 Step 200/349 | Loss: 1.3602
  Epoch 2 Step 250/349 | Loss: 1.3463
  Epoch 2 Step 300/349 | Loss: 1.3565
[bad clip] /orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips/fold_0_val/L4K0P3T4Q8_36_month_185_0_0.mp4: [13:57:48] /github/workspace/src/video/video_reader.cc:151: Check failed: st_nb >= 0 (-1381258232 vs. 0) ERROR cannot f

RuntimeError: DataLoader worker (pid(s) 1942401) exited unexpectedly

In [ ]:

# Aggregate cross-fold results

summary_rows = []
for fr in fold_results:
    summary_rows.append({
        'fold': fr['fold'],
        'top1_acc': fr['top1_acc'],
        'top2_acc': fr['top2_acc'],
        'macro_f1': fr['macro_f1'],
        'micro_f1': fr['micro_f1'],
        'weighted_f1': fr['weighted_f1'],
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df)
print('
Averages:')
print(summary_df.mean(numeric_only=True))


In [ ]:

# Inspect subset metrics

subset_summary = []
for fr in fold_results:
    for name, metrics in fr['subset_metrics'].items():
        row = {'fold': fr['fold'], 'subset': name}
        row.update(metrics)
        subset_summary.append(row)
subset_df = pd.DataFrame(subset_summary)
subset_df


In [ ]:

# Plot PR curves for a few top classes from the last fold

if fold_results:
    last_pr = fold_results[-1]['pr_curves']
    plt.figure(figsize=(8, 6))
    for cls, vals in last_pr.items():
        plt.plot(vals['recall'], vals['precision'], label=f"{cls} (AP={vals['ap']:.2f})")
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('PR Curves (validation, last fold)')
    plt.legend()
    plt.grid(True)
    plt.show()
